# OpTC Host ML — incremental streaming tree (river), shared shards

This notebook reads the same persisted 75/10/15 split and vocabulary as the other models.

This is the one model in the project that genuinely never holds more than one shard in memory,
for training *or* evaluation — it uses `river`'s single-pass, incremental Hoeffding Tree
(`learn_one`/`predict_proba_one` per row) instead of building any batch matrix. In exchange it
gives up two things the other notebooks have: (1) it evaluates the *complete* val/test streams —
this one only sees a randomly-selected, wall-clock-bounded subset of shards, because pure-Python
per-row updates cannot get through 3.48B/463M/695M rows inside a 12h Kaggle session; and (2) it is
architecturally simpler than a tuned batch forest/GBM, so expect it to underperform rf/lgbm/xgb on
raw metrics. Treat it as the "genuinely no batch materialization, honestly-reported coverage"
option, not a like-for-like replacement.


Coverage mechanism: a short warm-up measures real achieved rows/second on this hardware, then that
throughput is used to pick a random shard keep-probability for train/val/test so the whole run
stays inside configurable time budgets. Every shard is still decoded (cheap, vectorised numpy) —
only the expensive per-row Python loop is skipped for shards that are not selected — so the shards
that *are* processed are a representative random cross-section of the split, not just whichever
shards happen to come first in the manifest. Exact rows-seen and coverage fractions are written
into the metrics report so the partial-coverage caveat travels with the numbers.

Requires internet access enabled for this Kaggle notebook (Settings > Internet) to `pip install river`.


In [ ]:
import sys
!{sys.executable} -m pip install -q river


In [ ]:
from pathlib import Path
import sys, json
# Set an explicit path here only if several Host input datasets are attached.
MANIFEST = Path(
    '/kaggle/input/datasets/nhatnguyen2101/'
    'optc-host-ml-full-input/manifest.json'
)
if MANIFEST is None:
    candidates = []
    for path in Path('/kaggle/input').rglob('manifest.json'):
        try:
            meta = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if isinstance(meta, dict) and meta.get('format') == 'host_binary_delta_npz':
            candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Host dataset; set MANIFEST explicitly. Found: {candidates}')
    MANIFEST = candidates[0]
MANIFEST = Path(MANIFEST)
meta = json.loads(MANIFEST.read_text())
VOLUME_ROOTS = {key: MANIFEST.parent / 'volumes' / key for key in meta['volumes']}
sys.path.insert(0, str(MANIFEST.parent))
from host_shards import HostShardDataset
from shard_metrics import evaluate_stream
data = HostShardDataset(MANIFEST, VOLUME_ROOTS)
print('Manifest:', MANIFEST)
# Decode one shard per split to check the uploaded paths and format.
for split in ('train', 'val', 'test'):
    X_check, y_check = next(data.iter_batches(split))
    print(split, 'first shard:', X_check.shape, 'positive labels:', int(y_check.sum()))
    del X_check, y_check
print(json.dumps(data.meta['splits'], indent=2))
print('Full train CSR lower bound GiB:', data.csr_bytes('train') / 2**30)
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
MODEL_NAME = 'river_hoeffding'
import pickle, time
import numpy as np
from river import tree, imblearn

SEED = 1

def shard_rows(X, y):
    """Yield (feature_dict, label) pairs from a shard without densifying it.
    Only the nonzero (present) columns go into the dict -- river's dict-based
    inputs are a natural fit for this sparse, presence-only encoding."""
    indptr, indices = X.indptr, X.indices
    for i in range(X.shape[0]):
        start, end = indptr[i], indptr[i + 1]
        yield {int(c): 1.0 for c in indices[start:end]}, int(y[i])

def shard_feature_dicts(X):
    indptr, indices = X.indptr, X.indices
    for i in range(X.shape[0]):
        start, end = indptr[i], indptr[i + 1]
        yield {int(c): 1.0 for c in indices[start:end]}

class RandomSubsetDataset:
    """Wraps a HostShardDataset, passing a random subset of shards through.
    Every shard is still decoded -- only the per-row Python loop downstream is
    skipped for discarded shards, so kept shards stay a representative random
    cross-section of the whole split rather than a prefix of the manifest."""
    def __init__(self, dataset, keep_p, seed):
        self.dataset = dataset
        self.keep_p = keep_p
        self.rng = np.random.default_rng(seed)
    def iter_batches(self, split, **kwargs):
        for X, y in self.dataset.iter_batches(split, **kwargs):
            if self.rng.random() < self.keep_p:
                yield X, y

# RandomUnderSampler wraps the tree and online-undersamples the majority
# class per learn_one call, so the extreme (~0.006% positive) imbalance is
# handled the streaming way instead of via a batch resample.
model = imblearn.RandomUnderSampler(
    tree.HoeffdingTreeClassifier(grace_period=200, delta=1e-6, max_depth=12),
    desired_dist={0: 0.5, 1: 0.5}, seed=SEED)

# --- warm-up: measure real achieved rows/second on this hardware ---------
WARMUP_ROW_CAP = 200_000
WARMUP_TIME_CAP_SECONDS = 120
warmup_start = time.time()
warmup_rows = 0
for X_shard, y_shard in data.iter_batches('train'):
    for x, label in shard_rows(X_shard, y_shard):
        model.learn_one(x, label)
        warmup_rows += 1
    if warmup_rows >= WARMUP_ROW_CAP or time.time() - warmup_start > WARMUP_TIME_CAP_SECONDS:
        break
warmup_elapsed = time.time() - warmup_start
throughput = warmup_rows / max(warmup_elapsed, 1e-6)
print(f'warm-up: {warmup_rows} rows in {warmup_elapsed:.1f}s -> {throughput:.0f} rows/s')


In [ ]:
# --- budgeted training over a random cross-section of the remaining shards ---
TRAIN_TIME_BUDGET_SECONDS = 4 * 3600
train_rows_total = data.meta['splits']['train']['rows']
remaining_budget_rows = max(int(throughput * TRAIN_TIME_BUDGET_SECONDS) - warmup_rows, 0)
keep_p_train = min(1.0, remaining_budget_rows / max(train_rows_total - warmup_rows, 1))
print(f'estimated achievable rows in budget: {remaining_budget_rows} '
      f'(keep-probability over remaining shards: {keep_p_train:.6f})')

train_start = time.time()
rows_trained = warmup_rows
shards_seen = 0
for X_shard, y_shard in RandomSubsetDataset(data, keep_p_train, seed=SEED + 1).iter_batches('train'):
    shards_seen += 1
    for x, label in shard_rows(X_shard, y_shard):
        model.learn_one(x, label)
        rows_trained += 1
    if time.time() - train_start > TRAIN_TIME_BUDGET_SECONDS:
        break
elapsed_total = warmup_elapsed + (time.time() - train_start)
train_coverage = rows_trained / train_rows_total
print(f'trained on {rows_trained}/{train_rows_total} rows ({train_coverage:.4%} coverage) '
      f'in {elapsed_total/3600:.2f}h over {shards_seen} sampled shards (+ warm-up shards)')

predict = lambda X: np.clip(
    np.array([model.predict_proba_one(x).get(1, 0.0) for x in shard_feature_dicts(X)]), 0.0, 1.0)

with (OUTPUT / 'river_hoeffding_model.pkl').open('wb') as handle:
    pickle.dump(model, handle)


In [ ]:
# --- budgeted evaluation: a random, time-boxed subset of val/test each ---
# (NOT the complete stream -- see the markdown intro. If this raises
# "both classes required for evaluation", raise EVAL_TIME_BUDGET_SECONDS or
# TRAIN_TIME_BUDGET_SECONDS above and rerun; the positive class is rare
# enough that a very small time budget can occasionally miss it by chance.)
EVAL_TIME_BUDGET_SECONDS = 3600

def eval_keep_prob(split):
    total_rows = data.meta['splits'][split]['rows']
    budget_rows = int(throughput * EVAL_TIME_BUDGET_SECONDS)
    return min(1.0, budget_rows / max(total_rows, 1))

val_keep_p = eval_keep_prob('val')
print(f'val keep-probability: {val_keep_p:.6f} (~{EVAL_TIME_BUDGET_SECONDS/3600:.1f}h budget)')
validation = evaluate_stream(RandomSubsetDataset(data, val_keep_p, seed=SEED + 2), 'val', predict)

test_keep_p = eval_keep_prob('test')
print(f'test keep-probability: {test_keep_p:.6f} (~{EVAL_TIME_BUDGET_SECONDS/3600:.1f}h budget)')
test = evaluate_stream(RandomSubsetDataset(data, test_keep_p, seed=SEED + 3), 'test',
    predict, threshold=validation['threshold'])

report = {'model': MODEL_NAME, 'split_sha256':data.meta['split_sha256'],
    'vocabulary_sha256':data.meta['vocabulary_sha256'], 'seed':SEED,
    'streaming_coverage': {
        'measured_rows_per_second': throughput,
        'train_rows_seen': rows_trained, 'train_rows_total': train_rows_total,
        'train_coverage_fraction': train_coverage,
        'val_keep_probability': val_keep_p, 'test_keep_probability': test_keep_p,
        'note': ('val/test were evaluated on a random, wall-clock-bounded subset of shards, '
                 'not the full split. This differs from the rf/lgbm/xgb/isoforest notebooks, '
                 'which evaluate on the complete val/test streams.')},
    'validation':validation, 'test':test}
(OUTPUT / (MODEL_NAME + '_metrics.json')).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
